# L10 · RLHF-PPO end-to-end

## Goal

- SFT→reward→rollout→update를 잇는다
- policy와 frozen model의 소유권을 나눈다
- reward와 KL을 분해한다

## Setup

이 cell은 CPU·seed·offline 상태와 split hash를 먼저 고정합니다. toy 연산은 결정론적인 CPU 연산만 쓰며, package trainer의 전역 결정론 기본값은 유지합니다.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L10:toy:42").hexdigest()
print(f"lesson=L10 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L10 language=ko profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.12.13 rl_study=0.1.0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:7152104ceff29985d0262d3436bb57ad231f9f0d5822839b8bb374016cc72d1d data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. 현재 위치와 핵심 식

⏱ 5분 · 1/3 section · [필수/CORE]

현재 위치: LLM policy → **RLHF-PPO end-to-end** → 비교·평가

$$\text{SFT}\rightarrow\text{preference/RM}\rightarrow\text{rollout}\rightarrow\text{reward+KL}\rightarrow\text{PPO update}$$

RLHF-PPO에는 trainable policy와 value, frozen reference와 reward model이라는 서로 다른 역할이 있습니다. 이 toy는 deterministic verifier를 reward source로 써 pipeline을 완전히 offline으로 실행하지만 ownership과 mask 계약은 실제 모델과 같습니다.

### 2. 작은 숫자로 실행

⏱ 6분 · 2/3 section · [필수/CORE]

**먼저 예측:** PPO가 시작될 때 reference hash는 SFT policy의 어느 시점 hash와 같아야 하나요? 20초 동안 답을 적은 뒤 실행하세요.

<details><summary>정답 보기</summary>rollout 전에 복제한 초기 SFT policy hash와 같아야 하며 이후 고정됩니다.</details>

In [2]:
from rl_study.algorithms.rlhf_ppo import train_rlhf_ppo
from rl_study.algorithms.sft import train_sft
from rl_study.models.roles import parameter_sha256
sft_stage = train_sft(steps=2, batch_size=4, seed=42)
sft_hash = parameter_sha256(sft_stage.model)
rlhf_stage = train_rlhf_ppo(
    updates=1, batch_size=2, seed=42, policy=sft_stage.model,
    reward_source="verifier", update_epochs=1
)
print({"sft_steps": 2, "rlhf_updates": 1,
       "initial_sft_hash": sft_hash[:20],
       "generated_tokens": rlhf_stage.generated_tokens,
       "reference_hash": rlhf_stage.reference_hash[:20],
       "policy_loss": round(rlhf_stage.policy_losses[-1], 4)})

{'sft_steps': 2, 'rlhf_updates': 1, 'initial_sft_hash': 'sha256:6567e5680f133', 'generated_tokens': 44, 'reference_hash': 'sha256:6567e5680f133', 'policy_loss': 0.0}


### 3. 구현 해부

⏱ 6분 · 3/3 section · [심화/DEEP DIVE]

**왜 이렇게 구현했나:** 단계별 public API를 호출해 전체 lifecycle을 보되 trainer 내부를 notebook에 복사하지 않습니다. learned reward model은 대안이며 C5 구현에서 verifier와 동일한 interface를 공유합니다.

**흔한 함정:** reference가 policy와 함께 update되면 KL anchor가 움직여 penalty가 작아 보입니다. hash와 `requires_grad=False`, optimizer membership을 함께 검사합니다. 회귀 test: `test_rlhf_ppo_ratio_one_and_gradient_ownership`.

**쉬어가기:** 지금 출력한 한 값만 설명할 수 있으면 다음 cell로 가세요.

## Checks

In [3]:
assert rlhf_stage.generated_tokens > 0
assert rlhf_stage.reference_hash == sft_hash
print("checks=passed")

checks=passed


**회상 문제:** policy, value, reference, reward 네 역할 중 optimizer가 소유해야 하는 것은 무엇인가요? 1~2문장으로 답하세요.

## 내가 자주 틀리는 것

- loss가 유한하면 구현도 맞다고 생각한다.
- `terminated`와 `truncated`, prompt와 action을 합친다.
- 한 seed의 작은 결과를 알고리즘 순위로 확대한다.

## 60초 요약

- **실행 결론:** 2-step SFT 뒤 44 token을 rollout했고 reference hash가 초기 SFT hash와 일치했습니다. 한 update의 policy loss 0.0은 ratio=1 시작점 결과이지 학습 성공 주장이 아닙니다.
- 실제 확인: `test_rlhf_ppo_ratio_one_and_gradient_ownership`.
- 출력은 고정 seed의 toy 실행이며 논문 규모 결과가 아닙니다.

## Next Steps

1. L11에서는 online rollout 없이 chosen/rejected preference pair로 policy를 직접 최적화합니다.
2. `[필수/CORE]` assertion을 한 번 깨뜨리고 오류를 읽습니다.
3. package test를 열어 notebook의 작은 식과 production guard를 연결합니다.

[상세 구현 문서](../../docs/algorithms/rlhf-ppo.md) · [강좌 지도](../../docs/course-map.md)

## Sources

- `learning-to-summarize-2020` — `docs/sources.yml`
- `instructgpt-2022` — `docs/sources.yml`
- `repo-summarize-from-feedback` — `docs/sources.yml`